In [43]:
import pandas as pd
import os
from os.path import dirname


root_path = dirname(os.getcwd())

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/original/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/processed/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/graphs_repair/


In [44]:
dataset = "BPIC11_f2" # Select dataset to process

In [45]:
#if dataset == "bpic2012_O_DECLINED":
#    raw_data = pd.read_csv(f"datasets/original/{dataset}.csv")
#else:
raw_data = pd.read_csv(f"datasets/original/{dataset}.csv", sep=";")

raw_data.head()

,Diagnosis,Treatment code,Diagnosis code,Specialism code,Diagnosis Treatment Combination ID,Age,Case ID,label,Activity code,Producer code,Section,Specialism code.1,group,Number of executions,time:timestamp,timesincemidnight,month,weekday,hour,timesincelastevent,timesincecasestart,event_nr,open_cases
0,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,deviant,AC410100,SRTH,Section 5,SC61,Radiotherapy,1,2005-01-02 23:00:00,1380,1,6,23,0.0,0.0,1,5
1,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,deviant,AC419100,SRTH,Section 5,SC61,Radiotherapy,1,2005-01-02 23:00:00,1380,1,6,23,0.0,0.0,2,5
2,maligniteit cervix,TC3101,DC822,SC7,DTC153637,39,1,deviant,AC10113,SIOG,Section 1,SC13,Internal Specialisms clinic,1,2005-01-02 23:00:00,1380,1,6,23,0.0,0.0,1,5
3,maligniteit cervix,TC3101,DC822,SC7,DTC153637,39,1,deviant,AC410100,SIOG,Section 1,SC13,Internal Specialisms clinic,1,2005-01-02 23:00:00,1380,1,6,23,0.0,0.0,2,5
4,maligniteit cervix,TC3101,DC822,SC7,DTC153637,39,1,deviant,AC419100,SIOG,Section 1,SC13,Internal Specialisms clinic,1,2005-01-02 23:00:00,1380,1,6,23,0.0,0.0,3,5


In [46]:
if dataset == "BPIC11_f2" :
    tab_all = raw_data.rename(
        columns={"Case ID": "CaseID", "Activity code": "Activity","label":"Label"}
    )
elif dataset == "BPIC15_3_f2" or dataset == "BPIC17_O_Cancelled" or "sepsis_cases" in dataset:
    tab_all = raw_data.rename(
        columns={"Case ID": "CaseID","label":"Label"}
    )
elif dataset == "traffic_fines_1" or dataset == "hospital_billing_3" or "bpic2012" in dataset:
    tab_all = raw_data.rename(
        columns={"Case ID": "CaseID","label":"Label","Complete Timestamp":"time:timestamp"}
    )
else: 
    raise ValueError(f"Unknown dataset name: {dataset!r}")

In [47]:
from datetime import datetime
import time

def translate_time(time_str):
    return datetime.fromisoformat(time_str).timestamp()

In [48]:
tab_all["time:timestamp"] = tab_all["time:timestamp"].apply(translate_time)

In [49]:
tab_all.head()

,Diagnosis,Treatment code,Diagnosis code,Specialism code,Diagnosis Treatment Combination ID,Age,CaseID,Label,Activity,Producer code,Section,Specialism code.1,group,Number of executions,time:timestamp,timesincemidnight,month,weekday,hour,timesincelastevent,timesincecasestart,event_nr,open_cases
0,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,deviant,AC410100,SRTH,Section 5,SC61,Radiotherapy,1,1.104692e+09,1380,1,6,23,0.0,0.0,1,5
1,maligniteit cervix,TC103,M13,SC61,DTC376907,33,0,deviant,AC419100,SRTH,Section 5,SC61,Radiotherapy,1,1.104692e+09,1380,1,6,23,0.0,0.0,2,5
2,maligniteit cervix,TC3101,DC822,SC7,DTC153637,39,1,deviant,AC10113,SIOG,Section 1,SC13,Internal Specialisms clinic,1,1.104692e+09,1380,1,6,23,0.0,0.0,1,5
3,maligniteit cervix,TC3101,DC822,SC7,DTC153637,39,1,deviant,AC410100,SIOG,Section 1,SC13,Internal Specialisms clinic,1,1.104692e+09,1380,1,6,23,0.0,0.0,2,5
4,maligniteit cervix,TC3101,DC822,SC7,DTC153637,39,1,deviant,AC419100,SIOG,Section 1,SC13,Internal Specialisms clinic,1,1.104692e+09,1380,1,6,23,0.0,0.0,3,5


In [50]:
split_ratio = 4 / 5

first_act_tab = (
    tab_all.groupby("CaseID").first().sort_values("time:timestamp").reset_index()
)
first_act_tab = first_act_tab[
    ~first_act_tab.duplicated(subset=["CaseID", "Activity"], keep="first")
]
first_act_tab = first_act_tab.reset_index(drop=True)

list_train_valid_cases = list(
    first_act_tab[: int(split_ratio * len(first_act_tab))]["CaseID"].unique()
)

list_train_cases = list_train_valid_cases[: int(len(list_train_valid_cases) * 0.8)]
tab_train = tab_all[tab_all["CaseID"].isin(list_train_cases)].reset_index(drop=True)

list_valid_cases = list_train_valid_cases[int(len(list_train_valid_cases) * 0.8) :]
tab_valid = tab_all[tab_all["CaseID"].isin(list_valid_cases)].reset_index(drop=True)

list_test_cases = list(
    first_act_tab[int(split_ratio * len(first_act_tab)) :]["CaseID"].unique()
)
tab_test = tab_all[tab_all["CaseID"].isin(list_test_cases)].reset_index(drop=True)


#Find earliest timestamp in testing set.
split_ts = tab_test["time:timestamp"].min()

#Remove elements
tab_train = tab_train[tab_train["time:timestamp"] < split_ts].reset_index(drop=True)
tab_valid = tab_valid[tab_valid["time:timestamp"] < split_ts].reset_index(drop=True)




In [51]:
tab_all.to_csv(data_dir_processed + f"{dataset}_processed_all.csv", index=False)

In [52]:
tab_train.to_csv(data_dir_processed+ f"{dataset}_processed_train.csv", index = False)

In [53]:
tab_valid.to_csv(data_dir_processed+f"{dataset}_processed_valid.csv", index = False)

In [54]:
tab_test.to_csv(data_dir_processed+ f"{dataset}_processed_test.csv", index = False)